# 01 – Baseline Model Evaluation

Loads `baseline.pth`, evaluates on the **test set** (15%), prints all
segmentation metrics (mean + per-class Dice/IoU), saves a qualitative
prediction grid, and runs inference benchmarking.

In [ ]:
import sys, os
sys.path.insert(0, '/content/wet-amd-segmentation-edge/project')

import torch
from utils import Config, set_seed, get_device, device_info, print_dict
from utils.dataset import build_dataloaders
from models.model_loader import load_model, SegmentationInference
from models.baseline_model import CLASS_NAMES
from evaluation.metrics import compute_all_metrics, MetricAccumulator
from evaluation.benchmark import Benchmarker
from evaluation.visualization import plot_predictions, plot_per_class_dice

cfg = Config()
cfg.ensure_dirs()
set_seed(cfg.seed)

device = get_device()
print('Device:', device_info(device))

In [ ]:
# Load model
model = load_model(cfg, device=device)
inf   = SegmentationInference(model, device)   # no threshold — uses argmax
print(model)

In [ ]:
# Build dataloaders — 70 / 15 / 15 split
train_loader, val_loader, test_loader = build_dataloaders(cfg)
print(f'Train: {len(train_loader)} batches')
print(f'Val  : {len(val_loader)} batches')
print(f'Test : {len(test_loader)} batches')

In [ ]:
# Evaluate on test set
acc = MetricAccumulator()
sample_images, sample_gt, sample_pred = [], [], []

model.eval()
with torch.no_grad():
    for images, masks in test_loader:
        probs, pred_mask = inf.predict(images)   # pred_mask: (B, H, W) int64
        acc.update(compute_all_metrics(pred_mask, masks))
        if len(sample_images) < 4:
            sample_images.append(images)
            sample_gt.append(masks)
            sample_pred.append(pred_mask)

metrics = acc.mean()
print_dict(metrics, 'Baseline Test Set Metrics')

In [ ]:
# Per-class Dice summary
print('\n── Per-Class Dice ──────────────────────────────')
for name in CLASS_NAMES:
    key   = f"dice_{name.lower().replace(' ', '_')}"
    score = metrics.get(key, 0.0)
    bar   = '█' * int(score * 30)
    print(f'  {name:<18} {score:.4f}  {bar}')

In [ ]:
# Qualitative prediction grid (RGB colour-coded masks)
plot_predictions(
    torch.cat(sample_images),
    torch.cat(sample_gt),
    torch.cat(sample_pred),
    cfg, n=4, show=True
)

In [ ]:
# Per-class Dice bar chart
class_dice_scores = [
    metrics.get(f"dice_{n.lower().replace(' ', '_')}", 0.0)
    for n in CLASS_NAMES
]
plot_per_class_dice(class_dice_scores, cfg, show=True)

In [ ]:
# Inference benchmarking
bench   = Benchmarker(cfg, device)
results = bench.run(model, label='baseline_fp32')
Benchmarker.print_results(results)